# 01 · Data understanding
**Problem definition.** Estimate the probability that a patient has diabetes (Outcome = 1) from eight routine clinical measurements, as an *academic risk-prediction prototype*, not a diagnostic tool.

**Dataset.** Pima Indians Diabetes Database (NIDDK; CC0 on Kaggle). 768 women aged 21+ of Pima heritage. This notebook uses the same loading/validation code as the training script (`app/ml/data.py`).

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
BACKEND = (Path.cwd().parent / "backend").resolve()   # notebooks/ -> backend/
sys.path.insert(0, str(BACKEND))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from app.config import settings
pd.set_option("display.width", 140)

In [2]:
from app.ml.data import load_dataset, data_quality_report
from app.ml.features import FEATURES
df = load_dataset(settings.DATA_PATH)      # validates columns, types, target, negatives
print(df.shape)
df.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Feature dictionary (from `app/ml/features.py`, the single source of truth)

In [3]:
pd.DataFrame([{"column": f.csv_column, "api key": f.key, "unit": f.unit, "required": f.required,
               "0 means missing": f.zero_means_missing, "valid range": f"{f.min_value}–{f.max_value}"} for f in FEATURES])

,column,api key,unit,required,0 means missing,valid range
0,Pregnancies,pregnancies,count,True,False,0–20
1,Glucose,glucose,mg/dL,True,True,40–400
2,BloodPressure,blood_pressure,mmHg,True,True,20–140
3,SkinThickness,skin_thickness,mm,False,True,5–100
4,Insulin,insulin,µU/mL,False,True,10–900
5,BMI,bmi,kg/m²,True,True,12–70
6,DiabetesPedigreeFunction,diabetes_pedigree,score,True,False,0.05–3.0
7,Age,age,years,True,False,21–100


## Types, summary statistics and class balance

In [4]:
df.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [5]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.85,3.37,0.00,1.00,3.00,6.00,17.00
Glucose,768.0,120.89,31.97,0.00,99.00,117.00,140.25,199.00
BloodPressure,768.0,69.11,19.36,0.00,62.00,72.00,80.00,122.00
SkinThickness,768.0,20.54,15.95,0.00,0.00,23.00,32.00,99.00
Insulin,768.0,79.80,115.24,0.00,0.00,30.50,127.25,846.00
BMI,768.0,31.99,7.88,0.00,27.30,32.00,36.60,67.10
DiabetesPedigreeFunction,768.0,0.47,0.33,0.08,0.24,0.37,0.63,2.42
Age,768.0,33.24,11.76,21.00,24.00,29.00,41.00,81.00
Outcome,768.0,0.35,0.48,0.00,0.00,0.00,1.00,1.00


In [6]:
counts = df["Outcome"].value_counts().sort_index()
print(counts.to_dict(), f"-> {counts[1] / len(df):.1%} positive")

{0: 500, 1: 268} -> 34.9% positive


## Data quality
The minimum of Glucose, BloodPressure, SkinThickness, Insulin and BMI is **0**, which is physiologically impossible: zeros encode *not measured*. Pregnancies = 0 is a real value.

In [7]:
q = data_quality_report(df)
print("Duplicates:", q["n_duplicates"])
pd.DataFrame(q["zero_encoded_missing"]).T

Duplicates: 0


,count,pct
Glucose,5.0,0.65
BloodPressure,35.0,4.56
SkinThickness,227.0,29.56
Insulin,374.0,48.70
BMI,11.0,1.43


In [8]:
print("Rows with at least one missing value:", q["rows_with_any_missing"], "of", q["n_rows"])

Rows with at least one missing value: 376 of 768


**Takeaways**
- No duplicates; target is moderately imbalanced (≈35% positive) → use *stratified* splits and report PR-AUC as well as ROC-AUC.
- Insulin is missing for almost half the rows and SkinThickness for ~30% → impute **inside** the pipeline and add missing-value indicators.
- The small, single-population sample limits generalisation; this is stated in the app.